# STAT 547E — Assignment 2: Starter Notebook
### Problem 3: ELE failure and the cost of poor mixing

Submit a single PDF with all plots and written answers.

In [ ]:
using Distributions, Plots, Statistics, Random
gr()

In [ ]:
log_incr_weight(x, Δβ, μ) = Δβ .* (μ .* x .- μ^2 / 2)

function logsumexp(v)
    m = maximum(v)
    return m + log(sum(exp.(v .- m)))
end

"Systematic resampling. Returns N indices proportional to weights w."
function systematic_resample(w, N, rng)
    cumw = cumsum(w ./ sum(w))
    u    = (rand(rng) .+ (0:N-1)) ./ N
    idx  = zeros(Int, N)
    j    = 1
    for i in 1:N
        while cumw[j] < u[i]; j += 1; end
        idx[i] = j
    end
    return idx
end

"AIS estimate of log Z under ELE (exact sampling at each temperature)."
function ais_ele(μ, T, N; rng=default_rng())
    βs   = range(0.0, 1.0, length=T+1)
    logW = zeros(N)
    for t in 1:T
        X     = randn(rng, N) .+ βs[t] * μ
        logW .+= log_incr_weight(X, βs[t+1] - βs[t], μ)
    end
    return logsumexp(logW) - log(N)
end

"SIR estimate of log Z under ELE with systematic resampling."
function sir_ele(μ, T, N; rng=default_rng())
    βs   = range(0.0, 1.0, length=T+1)
    logZ = 0.0
    for t in 1:T
        X    = randn(rng, N) .+ βs[t] * μ
        logw = log_incr_weight(X, βs[t+1] - βs[t], μ)
        logZ += logsumexp(logw) - log(N)
    end
    return logZ
end

"AIS with k RWM steps per temperature."
function ais_rwm(μ, T, N, k, σ_prop; rng=default_rng())
    βs   = range(0.0, 1.0, length=T+1)
    X    = randn(rng, N)
    logW = zeros(N)
    for t in 1:T
        logW .+= log_incr_weight(X, βs[t+1] - βs[t], μ)
        for _ in 1:k
            X_prop = X .+ σ_prop .* randn(rng, N)
            log_α  = -0.5*(X_prop .- βs[t+1]*μ).^2 .+
                      0.5*(X      .- βs[t+1]*μ).^2
            accept = log.(rand(rng, N)) .< log_α
            X[accept] = X_prop[accept]
        end
    end
    return logsumexp(logW) - log(N)
end

"SIR with k RWM steps per temperature and systematic resampling."
function sir_rwm(μ, T, N, k, σ_prop; rng=default_rng())
    βs   = range(0.0, 1.0, length=T+1)
    X    = randn(rng, N)
    logZ = 0.0
    for t in 1:T
        logw = log_incr_weight(X, βs[t+1] - βs[t], μ)
        logZ += logsumexp(logw) - log(N)
        w    = exp.(logw .- maximum(logw))
        idx  = systematic_resample(w, N, rng)
        X    = X[idx]
        for _ in 1:k
            X_prop = X .+ σ_prop .* randn(rng, N)
            log_α  = -0.5*(X_prop .- βs[t+1]*μ).^2 .+
                      0.5*(X      .- βs[t+1]*μ).^2
            accept = log.(rand(rng, N)) .< log_α
            X[accept] = X_prop[accept]
        end
    end
    return logZ
end

"Run est_fn M times at each T in T_grid. Returns (means, stds) of log Ẑ."
function logz_stats(est_fn, T_grid, M)
    means, stds = Float64[], Float64[]
    for T in T_grid
        runs = [est_fn(T) for _ in 1:M]
        push!(means, mean(runs))
        push!(stds,  std(runs))
    end
    return means, stds
end

const M_REP = 20   # 20 replicates — enough to see the band shape


In [ ]:
log_incr_weight(x, Δβ, μ) = Δβ .* (μ .* x .- μ^2 / 2)

function logsumexp(v)
    m = maximum(v)
    return m + log(sum(exp.(v .- m)))
end

"Systematic resampling. Returns N indices proportional to weights w."
function systematic_resample(w, N, rng)
    cumw = cumsum(w ./ sum(w))
    u    = (rand(rng) .+ (0:N-1)) ./ N
    idx  = zeros(Int, N)
    j    = 1
    for i in 1:N
        while cumw[j] < u[i]; j += 1; end
        idx[i] = j
    end
    return idx
end

"AIS estimate of log Z under ELE (exact sampling at each temperature)."
function ais_ele(μ, T, N; rng=default_rng())
    βs   = range(0.0, 1.0, length=T+1)
    logW = zeros(N)
    for t in 1:T
        X     = randn(rng, N) .+ βs[t] * μ
        logW .+= log_incr_weight(X, βs[t+1] - βs[t], μ)
    end
    return logsumexp(logW) - log(N)
end

"SIR estimate of log Z under ELE with systematic resampling."
function sir_ele(μ, T, N; rng=default_rng())
    βs   = range(0.0, 1.0, length=T+1)
    logZ = 0.0
    for t in 1:T
        X    = randn(rng, N) .+ βs[t] * μ
        logw = log_incr_weight(X, βs[t+1] - βs[t], μ)
        logZ += logsumexp(logw) - log(N)
    end
    return logZ
end

"AIS with k RWM steps per temperature."
function ais_rwm(μ, T, N, k, σ_prop; rng=default_rng())
    βs   = range(0.0, 1.0, length=T+1)
    X    = randn(rng, N)
    logW = zeros(N)
    for t in 1:T
        logW .+= log_incr_weight(X, βs[t+1] - βs[t], μ)
        for _ in 1:k
            X_prop = X .+ σ_prop .* randn(rng, N)
            log_α  = -0.5*(X_prop .- βs[t+1]*μ).^2 .+
                      0.5*(X      .- βs[t+1]*μ).^2
            accept = log.(rand(rng, N)) .< log_α
            X[accept] = X_prop[accept]
        end
    end
    return logsumexp(logW) - log(N)
end

"SIR with k RWM steps per temperature and systematic resampling."
function sir_rwm(μ, T, N, k, σ_prop; rng=default_rng())
    βs   = range(0.0, 1.0, length=T+1)
    X    = randn(rng, N)
    logZ = 0.0
    for t in 1:T
        logw = log_incr_weight(X, βs[t+1] - βs[t], μ)
        logZ += logsumexp(logw) - log(N)
        w    = exp.(logw .- maximum(logw))
        idx  = systematic_resample(w, N, rng)
        X    = X[idx]
        for _ in 1:k
            X_prop = X .+ σ_prop .* randn(rng, N)
            log_α  = -0.5*(X_prop .- βs[t+1]*μ).^2 .+
                      0.5*(X      .- βs[t+1]*μ).^2
            accept = log.(rand(rng, N)) .< log_α
            X[accept] = X_prop[accept]
        end
    end
    return logZ
end

"Run est_fn M times at each T in T_grid. Returns (means, stds) of log Ẑ."
function logz_stats(est_fn, T_grid, M)
    means, stds = Float64[], Float64[]
    for T in T_grid
        runs = [est_fn(T) for _ in 1:M]
        push!(means, mean(runs))
        push!(stds,  std(runs))
    end
    return means, stds
end

const M_REP = 20   # 20 replicates — enough to see the band shape


---
## Part (a): AIS — varying $k$

**Write your observations here.**

- At what $k$ does the band converge to the ELE baseline?
- By what factor in $T$ does $k=1$ inflate the required schedule?

---
## Part (b): SIR — varying $k$

**Write your observations here.**

- Does resampling change the $k$ at which ELE kicks in?
- At small $k$, is the SIR band wider or narrower than AIS?

---
## Part (c): Equal budget — few steps + good mixing vs many steps + poor mixing

**Write your observations here.**

- Which strategy gives a better estimate at the same budget $B = T \times k = 2000$?
- Is the advantage larger for AIS or SIR? Why?